# 07 — Docker, containers & Azure Container Apps

**Objectif :** comprendre les **containers** (Docker), les **registres d'images** (ACR) et la plateforme **Azure Container Apps** (ACA) qui orchestre tout ça. À la fin, on **build et déploie un mini container Hello World** sur ACA.

C'est l'avant-dernière brique avant d'attaquer le projet `mailroom-ia` qui utilise précisément ACA Environment + Apps + Jobs.

À l'issue, vous saurez :
- ce qu'est un **container** vs une VM, et pourquoi c'est devenu le standard ;
- lire et écrire un **Dockerfile** simple (multi-stage) ;
- ce qu'est une **image**, un **registry** (Docker Hub, **Azure Container Registry**) ;
- la différence **Container App** vs **Container App Job** (Apps & Jobs) ;
- ce qu'est **KEDA** et comment ça scale event-driven ;
- déployer concrètement un container sur ACA en quelques commandes.

> 🔜 Le **notebook 08** enchaîne sur **Bicep / Infrastructure as Code** et contient le **cleanup final** de tout le parcours.

> ⚠️ Ce notebook **utilise Docker en local** pour la partie hands-on (sections 4 et 8). Si vous êtes en Codespaces : Docker est déjà dispo. Si vous êtes en local Windows : il faut **Docker Desktop** installé.

## 1. Container vs VM — la différence en 1 schéma

```
  VIRTUAL MACHINE                          CONTAINER
  ───────────────                          ─────────
  ┌─────────────────────────┐              ┌─────────────────────────┐
  │  App                    │              │  App                    │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Libs / runtime         │              │  Libs / runtime         │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  OS guest (Linux/Windows)│              │ (rien — partage du noyau)│
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Hyperviseur            │              │  Container runtime      │
  ├─────────────────────────┤              ├─────────────────────────┤
  │  OS hôte                │              │  OS hôte (noyau partagé)│
  ├─────────────────────────┤              ├─────────────────────────┤
  │  Hardware               │              │  Hardware               │
  └─────────────────────────┘              └─────────────────────────┘
       lourd, ~GB                              léger, ~MB
       démarre en min                          démarre en sec
       isolation totale                        isolation processus
```

Un **container** = un processus isolé (cgroups + namespaces Linux) qui embarque son code + ses libs, **sans OS complet**. Il partage le noyau de l'hôte.

Conséquences :
- **Démarre en quelques secondes** (vs minutes pour une VM)
- **Image ~50-300 MB** (vs plusieurs GB pour une VM)
- **Densité** : 50 containers/VM sans souci
- **Portable** : la même image tourne sur votre PC, en CI, en prod
- **Immuable** : on ne *modifie* pas un container, on en *recrée* un autre

## 2. Vocabulaire à connaître

| Terme | Définition |
|-------|------------|
| **Image** | Le « modèle figé » : code + libs + config. Read-only. Identifiée par `nom:tag` (ex. `python:3.13-slim`). |
| **Container** | Une **instance** d'une image qui tourne. On peut en lancer N depuis la même image. |
| **Dockerfile** | Recette texte pour fabriquer une image, étape par étape (cf. §3). |
| **Layer** | Une étape Dockerfile = une couche cacheable. Les images partagent leurs couches communes. |
| **Registry** | Un dépôt d'images. Public (**Docker Hub**, `mcr.microsoft.com`) ou privé (**Azure Container Registry**, GitHub Container Registry…). |
| **Tag** | Un label sur une image (`v1`, `latest`, `2.5.0`). **`latest` ne veut rien dire** — toujours pinner. |
| **Digest** | Hash SHA-256 immuable de l'image (`sha256:abcd...`). C'est le vrai identifiant. |
| **Engine / runtime** | Le programme qui exécute les containers : Docker, containerd, Podman… |
| **OCI** | Le standard ouvert (Open Container Initiative) qui définit le format image + runtime. Toutes les implémentations sont compatibles. |

## 3. Anatomie d'un Dockerfile

Voici le **Dockerfile multi-stage** du worker du projet `mailroom-ia` (simplifié) :

```dockerfile
# syntax=docker/dockerfile:1.7

# ─── STAGE 1 : builder ───
FROM python:3.13-slim AS builder
WORKDIR /build
RUN pip install --no-cache-dir uv
COPY pyproject.toml ./
RUN uv pip compile pyproject.toml -o requirements.txt

# ─── STAGE 2 : runtime ───
FROM python:3.13-slim
WORKDIR /app

# Bonnes pratiques Python conteneurisé
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1

# Utilisateur non-root (sécurité)
RUN groupadd -r app && useradd -r -g app -d /app app

# Copie depuis le stage builder
COPY --from=builder /build/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY worker/ ./worker/
USER app

CMD ["python", "-m", "worker.main"]
```

**Pourquoi multi-stage ?**
- Le `builder` peut contenir des outils lourds (compilateurs, dev deps) ;
- L'image finale ne garde que **le strict nécessaire** au runtime.
- Résultat : image **2-5× plus petite**, surface d'attaque réduite.

**Bonnes pratiques (à mémoriser)** :

| Règle | Pourquoi |
|-------|----------|
| Image de base **`-slim`** ou **`-alpine`** | Plus petite, moins de CVE |
| Ne **pas tourner en root** (`USER app`) | Sécurité — moindre privilège |
| **Pin** les versions (`python:3.13-slim`, pas `python:latest`) | Reproductibilité |
| `.dockerignore` propre (exclure `.git`, `node_modules`, etc.) | Build plus rapide, image plus petite |
| Ordre des `COPY` : du plus stable au plus changeant | Optimise le cache des couches |
| **Healthcheck** côté Dockerfile ou côté orchestrateur (ACA) | Détection de pannes |

## 4. Hands-on local — build, run, inspect

On va builder un mini Hello-World en Python directement dans une cellule. On crée le Dockerfile à la volée.

In [ ]:
# Crée un répertoire de travail temporaire pour le mini container
from pathlib import Path
DEMO = Path("./_docker-demo")
DEMO.mkdir(exist_ok=True)

(DEMO / "app.py").write_text(
    "import os, sys\n"
    "print(f'Hello depuis {sys.platform} Python {sys.version.split()[0]}')\n"
    "print(f'Hostname container : {os.uname().nodename}')\n",
    encoding="utf-8",
)

(DEMO / "Dockerfile").write_text(
    "FROM python:3.13-slim\n"
    "WORKDIR /app\n"
    "COPY app.py .\n"
    'CMD ["python", "app.py"]\n',
    encoding="utf-8",
)

print("✓ Dockerfile + app.py créés dans", DEMO.resolve())

In [ ]:
# Vérifier que Docker tourne
!docker version

In [ ]:
# Build l'image (≈ 30 sec la première fois)
!docker build -t hello-stage:v1 ./_docker-demo

In [ ]:
# Lister mes images locales
!docker images hello-stage

In [ ]:
# Lancer le container (instance de l'image)
!docker run --rm hello-stage:v1

In [ ]:
# Inspecter les couches (layers) de l'image
!docker history hello-stage:v1

## 5. Les registres — où vivent les images

Une image construite localement n'est dispo que sur **votre machine**. Pour qu'un cluster Azure puisse la déployer, il faut la **pousser dans un registre**.

| Registre | Usage |
|----------|-------|
| **Docker Hub** | Public par défaut. Beaucoup d'images de base (`python`, `node`…). Limites pull pour utilisateurs anonymes. |
| **`mcr.microsoft.com`** | Microsoft Container Registry — images Microsoft (devcontainers, .NET, SQL Server…). Public, sans limite. |
| **GitHub Container Registry** (`ghcr.io`) | Lié à GitHub Actions, gratuit, populaire en CI/CD. |
| **Azure Container Registry (ACR)** ⭐ | Registry privé Azure, intégré RBAC + Managed Identity. **C'est ce qu'on utilise pour le projet `mailroom-ia`.** |

### Le flow standard

```
  docker build → image locale
        │
        ▼
  docker tag <local> <registry>/<repo>:<tag>
        │
        ▼
  docker login <registry>     ← ou MI / OIDC en CI
        │
        ▼
  docker push <registry>/<repo>:<tag>
        │
        ▼
  L'image est dispo dans le registry → ACA / AKS / autre orchestrateur peut la pull
```

### `az acr build` — build cloud-side

Pour éviter d'avoir Docker installé localement, **Azure peut builder pour vous** :

```bash
az acr build --registry <acr-name> --image hello:v1 .
```

→ ACR fait le `docker build` côté serveur, dans un build agent managé, et stocke l'image. C'est ce qu'on utilise dans le `setup.ipynb` du projet.

## 6. Azure Container Apps (ACA)

ACA est **la plateforme PaaS Azure pour exécuter des containers** sans gérer de Kubernetes. C'est ce qu'utilise le projet `mailroom-ia` (cf. notebook setup côté projet).

### Hiérarchie

```
  Azure Container Apps Environment       ← le « cluster logique » (un par projet/env)
      │   - réseau VNet partagé
      │   - Log Analytics commun
      │   - Dapr disponible
      │
      ├── Container App   web            ← HTTP ingress, long-running
      │     - scale 1→N sur trafic HTTP
      │     - Managed Identity
      │
      ├── Container App Job  worker      ← event-driven (queue), exécution finie
      │     - scale 0→N via KEDA
      │     - 1 message = 1 exécution
      │
      └── Container App Job  cron-clean  ← scheduled (cron) optionnel
```

### Apps vs Jobs — quand utiliser quoi ?

| | **Container App** | **Container App Job** |
|---|-------------------|-----------------------|
| Modèle | Long-running, **toujours up** (au moins 1 replica si `minReplicas≥1`) | **Exécutions finies** : la job démarre, fait son truc, s'arrête |
| Trigger | HTTP / TCP / event sur replica | **Manual** / **Schedule** (cron) / **Event** (KEDA) |
| Scaling | KEDA + HTTP scaler | KEDA (event-driven) ou schedule |
| Idéal pour | API web, BFF, frontend | Worker queue, batch, ETL, cron |
| Coût | Replica idle = facturée | **0 € si rien ne tourne** |

**Dans `mailroom-ia`** : `web` = Container App (Next.js HTTP), `worker-classify` = Container App **Job** event-driven (lance 1 execution par message Storage Queue).

## 7. KEDA — *Kubernetes Event Driven Autoscaler*

**KEDA** est l'autoscaler open-source qui équipe nativement ACA. Il sait scaler sur **50+ types d'événements** :

| Type d'événement | Scaler KEDA |
|-----------------|-------------|
| Longueur d'une **Storage Queue** Azure | `azure-queue` |
| Messages **Service Bus** | `azure-servicebus` |
| Lag d'un topic **Kafka** | `kafka` |
| Charge CPU / mémoire | `cpu`, `memory` |
| Cron (calendrier) | `cron` |
| Requêtes HTTP/sec | `http` (Container Apps natif) |
| Métrique custom Prometheus | `prometheus` |
| … | … |

### Exemple — le scaler du worker `mailroom-ia`

Tiré du Bicep `infra-apps.bicep` du projet :

```yaml
scale:
  minExecutions: 0      # scale-to-zero quand rien à faire
  maxExecutions: 10     # cap à 10 executions parallèles
  pollingInterval: 30   # KEDA vérifie la queue toutes les 30 s
  rules:
    - name: queue-length
      type: azure-queue
      metadata:
        accountName: stmailroomXXX
        queueName: doc-to-classify
        queueLength: '1'   # 1 message = 1 execution déclenchée
      identity: system     # ← KEDA s'authentifie via la Managed Identity du Job
```

> 🔐 **Important** : KEDA lit la queue via la **Managed Identity** du Job (pas via une connection string avec clé). C'est pour ça que dans `infra-apps.bicep` on assigne le rôle `Storage Queue Data Message Processor` à l'identité du worker.

## 8. Hands-on — déployer notre mini container sur ACA

On va :
1. Push l'image `hello-stage:v1` qu'on a buildée en §4 vers un nouvel ACR éphémère
2. Créer un ACA Environment
3. Déployer un Container App qui tourne notre image

👇 **Action requise** : remplacez par votre RG (la même variable que les autres notebooks).

In [ ]:
import os, re, subprocess

RG = "rg-stage-<votre-identifiant>"
m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")

LOCATION = subprocess.check_output(
    f"az group show --name {RG} --query location -o tsv", shell=True
).decode().strip()

clean = USER_ID.replace("-", "")
ACR_NAME = f"acrhello{clean}"
ACA_ENV = f"acaenv-hello-{USER_ID}"
ACA_APP = f"hello-{USER_ID}"

for k, v in {"RG": RG, "USER_ID": USER_ID, "LOCATION": LOCATION,
             "ACR_NAME": ACR_NAME, "ACA_ENV": ACA_ENV, "ACA_APP": ACA_APP}.items():
    os.environ[k] = v

print(f"RG       : {RG}")
print(f"ACR      : {ACR_NAME}")
print(f"ACA env  : {ACA_ENV}")
print(f"ACA app  : {ACA_APP}")

In [ ]:
# Installer l'extension containerapp + enregistrer les providers (idempotent)
!az extension add --name containerapp --upgrade --only-show-errors
!az provider register --namespace Microsoft.App --wait
!az provider register --namespace Microsoft.OperationalInsights --wait

In [ ]:
# 1. Créer un ACR éphémère et y builder l'image (cloud-side, pas besoin de Docker local)
!az acr create --name $ACR_NAME --resource-group $RG --sku Basic --output table
!az acr build --registry $ACR_NAME --image hello-stage:v1 ./_docker-demo

In [ ]:
# 2. Créer l'ACA Environment
!az containerapp env create \
    --name $ACA_ENV \
    --resource-group $RG \
    --location $LOCATION \
    --output table

In [ ]:
# 3. Déployer le Container App
#    On utilise --system-assigned pour activer la Managed Identity
#    et --registry-identity system pour pull via cette identité (pas de mot de passe ACR).
!az containerapp create \
    --name $ACA_APP \
    --resource-group $RG \
    --environment $ACA_ENV \
    --image "$ACR_NAME.azurecr.io/hello-stage:v1" \
    --registry-server "$ACR_NAME.azurecr.io" \
    --system-assigned \
    --registry-identity system \
    --min-replicas 0 --max-replicas 1 \
    --cpu 0.25 --memory 0.5Gi \
    --output table

In [ ]:
# 4. Voir les logs (le container a tourné, affiché Hello, et s'est terminé)
!az containerapp logs show --name $ACA_APP --resource-group $RG --tail 50

🎉 Vous venez de **builder, push et déployer un container sur Azure Container Apps**. C'est exactement le pattern qu'on industrialise dans le projet `mailroom-ia` — en plus gros.

## 9. Pour cleaner ce notebook

On garde l'ACA env + l'ACR créés ici pour le **notebook 08** (on va les réutiliser pour démontrer Bicep). Le cleanup final du parcours arrive dans le 08.

Si vous voulez quand même supprimer maintenant les ressources créées par ce notebook :

In [ ]:
# Voir ce qui existe dans le RG
!az resource list --resource-group $RG --output table

In [ ]:
# Cleanup du Container App seul (optionnel)
# Décommentez si vous voulez juste supprimer l'app, en gardant l'ACA env et l'ACR pour le notebook 08.

# !az containerapp delete --name $ACA_APP --resource-group $RG --yes

print("💡 Astuce : laissez tout en place pour le notebook 08 (Bicep) qui va réutiliser l'ACA env.")

## Récap

Vous savez maintenant :
- ce qu'est un **container** vs une VM ;
- lire et écrire un **Dockerfile** simple (multi-stage, non-root, pin de version) ;
- ce qu'est un **registry** (Docker Hub, MCR, ACR) ;
- la différence **Container App** vs **Container App Job** ;
- ce qu'est **KEDA** et comment ça scale event-driven ;
- déployer un container sur ACA avec Managed Identity et pull ACR sans secret.

### 🚀 Prochaine étape : **notebook 08 — Bicep & Infrastructure as Code**

Vous avez tout déployé jusque-là avec des commandes `az` impératives. Le notebook 08 va vous apprendre à **décrire votre infra en code Bicep** et à la déployer en une commande. C'est le même langage qu'on utilise dans le projet `mailroom-ia` (cf. `projet/mailroom-ia/infra/`).

Le **cleanup final** du parcours sera également dans le notebook 08.

📚 Pour aller plus loin :
- Doc ACA : https://learn.microsoft.com/azure/container-apps/
- Doc KEDA : https://keda.sh/docs/
- Best practices Docker : https://docs.docker.com/develop/develop-images/dockerfile_best-practices/
- ACR : https://learn.microsoft.com/azure/container-registry/